In [157]:
# ============================================================
# NOTEBOOK 02 — Sequence Baseline (CNN)
# Promoter Sequence Classification Baseline
# ============================================================

In [158]:
# !pip install torch torchvision torchaudio -q

In [159]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

In [160]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [161]:
df = pd.read_csv("nodes_promoters.csv")

X = np.load("promoter_sequences_onehot.npy")

print(df.head())
print(X.shape)

               pmId  pmName   strand     posTSS sigmaFactor  \
0  RDBECOLIPMC00001    spyp  reverse  1825688.0     sigma70   
1  RDBECOLIPMC00002   yfeKp  forward  2537315.0     sigma24   
2  RDBECOLIPMC00003  bepAp1  forward  2616068.0     sigma24   
3  RDBECOLIPMC00004   bamBp  reverse  2638975.0     sigma24   
4  RDBECOLIPMC00005   recCp  reverse  2962507.0     sigma70   

                                          pmSequence  
0  acactttcattgttttaccgttgctctgattaattgacgctaaagt...  
1  ccgatgatcctcatcgtaatccaaccgaaactttacctgattctgg...  
2  gccgttacactcaaaggcggcgcggtgggaacgatatttcacagta...  
3  aaatacttatggtgcgctggcttctttggaacttgcgcagcaattt...  
4  tgccaactggcaggtcaaccgaatgcagacatcgcaggcgggatgt...  
(3964, 4, 81)


In [162]:
print(df.columns)

Index(['pmId', 'pmName', 'strand', 'posTSS', 'sigmaFactor', 'pmSequence'], dtype='object')


In [163]:
mask = df["sigmaFactor"].notna() & df["pmSequence"].notna()

df = df[mask].reset_index(drop=True)

X = np.load("promoter_sequences_onehot.npy")

X = X[mask.values]

df["sigmaFactor"] = df["sigmaFactor"].astype(str)

labels = sorted(df["sigmaFactor"].unique())

label_map = {label: idx for idx, label in enumerate(labels)}

y = df["sigmaFactor"].map(label_map).values

print(X.shape)
print(y.shape)

(3316, 4, 81)
(3316,)


In [164]:
y = df["sigmaFactor"].astype("category").cat.codes.values

print(y.shape)

(3316,)


In [165]:
print(X.shape)
print(y.shape)

(3316, 4, 81)
(3316,)


In [166]:
# X = np.load("promoter_sequences_onehot.npy")

# y = df["sigmaFactor"].astype("category").cat.codes.values

# print(X.shape)
# print(y.shape)

In [167]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,

)

In [168]:
class SequenceDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SequenceDataset(X_train, y_train)
test_dataset = SequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [169]:
class CNNBaseline(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=4,
            out_channels=32,
            kernel_size=5,
            padding=2
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(
            32,
            64,
            kernel_size=5,
            padding=2
        )

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(64 * 20, 128)

        self.dropout = nn.Dropout(0.3)

        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):


        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [170]:
num_classes = len(np.unique(y))

model = CNNBaseline(num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

CNNBaseline(
  (conv1): Conv1d(4, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=1280, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)


In [171]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in tqdm(train_loader):

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

100%|██████████| 83/83 [00:01<00:00, 65.08it/s]


Epoch 1/10 | Loss: 1.2857


100%|██████████| 83/83 [00:00<00:00, 85.01it/s]


Epoch 2/10 | Loss: 1.0906


100%|██████████| 83/83 [00:00<00:00, 85.53it/s]


...

Epoch 8/10 | Loss: 0.5209


100%|██████████| 83/83 [00:01<00:00, 53.33it/s]


Epoch 9/10 | Loss: 0.4706


100%|██████████| 83/83 [00:00<00:00, 128.04it/s]

Epoch 10/10 | Loss: 0.4235


In [172]:
model.eval()

predictions = []
targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        predictions.extend(preds)

        targets.extend(y_batch.numpy())

acc = accuracy_score(targets, predictions)

f1 = f1_score(targets, predictions, average="weighted")

print("\n===== RESULTS =====")
print("Accuracy:", acc)
print("F1 Score:", f1)


===== RESULTS =====
Accuracy: 0.786144578313253
F1 Score: 0.7428524635480529


In [173]:
torch.save(model.state_dict(), "cnn_sequence_baseline.pt")

print("Model saved.")

Model saved.


In [174]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy:", accuracy_score(targets, predictions))

print("Precision (macro):",
      precision_score(targets, predictions, average="macro"))

print("Recall (macro):",
      recall_score(targets, predictions, average="macro"))

print("F1 Score (macro):",
      f1_score(targets, predictions, average="macro"))

print("\n===== CLASSIFICATION REPORT =====")

print(classification_report(targets, predictions))

print("\n===== CONFUSION MATRIX =====")

print(confusion_matrix(targets, predictions))

Accuracy: 0.786144578313253
Precision (macro): 0.6809978183809959
Recall (macro): 0.5290346543221881
F1 Score (macro): 0.5276060670682047

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           1       0.86      0.79      0.83       116
           2       0.73      0.59      0.65        27
           3       0.71      0.81      0.76        68
           4       1.00      0.04      0.07        53
           5       0.00      0.00      0.00        21
           6       0.78      0.94      0.86       379

    accuracy                           0.79       664
   macro avg       0.68      0.53      0.53       664
weighted avg       0.78      0.79      0.74       664


===== CONFUSION MATRIX =====
[[ 92   0   5   0   0  19]
 [  0  16   3   0   0   8]
 [  0   1  55   0   0  12]
 [  2   0   2   2   1  46]
 [  5   1   2   0   0  13]
 [  8   4  10   0   0 357]]


In [175]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score

# Get probabilities
model.eval()

all_probs = []

with torch.no_grad():

    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probs = torch.softmax(outputs, dim=1)

        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)

# Convert labels to one-hot format
y_test_bin = label_binarize(
    targets,
classes=range(all_probs.shape[1])
)

# Multiclass AUC-ROC
auc_macro = roc_auc_score(
    y_test_bin,
    all_probs,
    multi_class="ovr",
    average="macro"
)

auc_weighted = roc_auc_score(
    y_test_bin,
    all_probs,
    multi_class="ovr",
    average="weighted"
)

print("AUC-ROC Macro:", auc_macro)
print("AUC-ROC Weighted:", auc_weighted)

AUC-ROC Macro: nan
AUC-ROC Weighted: 0.9194925602857527


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
